# Chapter 12: Data Structures for Graphics

**Source span.** This notebook is grounded in *Fundamentals of Computer Graphics*, Chapter 12, printed pages 291-334 and physical PDF pages 308-351. The source span introduces triangle mesh storage and topology, triangle strips and fans, triangle-neighbor, winged-edge, and half-edge connectivity, scene graphs, ray-tracing instancing, bounding boxes, BVHs, uniform grids, axis-aligned and polygon-aligned BSP trees, and tiled multidimensional arrays.

**Chapter goal.** Learn to read a graphics data structure by the questions it can answer quickly: What is adjacent to this triangle? What world transform applies to this object? Which objects can a ray possibly hit? Which side of a splitting plane contains this primitive? Which memory locations are close during a scan?

The chapter is not about one universal structure. It is about matching the representation to the query. The notebook keeps every example small enough to audit: a closed triangle mesh, a ferry/car scene graph, a disc scene for ray traversal, a triangle cut by a BSP plane, and a padded tiled array. Each section records both a visual artifact and an invariant that would catch a common implementation bug.

## Translation guide

| Book concept | Computational translation used here | Failure or invariant check |
| --- | --- | --- |
| Triangle mesh topology | Vertices, triangular faces, undirected edge incidence, directed half-edges | Euler count, every closed edge used twice, pair and next pointers are reciprocal, one-ring traversal closes |
| Indexed meshes, strips, and fans | Vertex array plus index streams | Storage units compare separate triangles, indexed triangles, strips/fans, and richer connectivity structures |
| Triangle-neighbor, winged-edge, half-edge structures | Local state machines for crossing from face to face or around a vertex | Traversal touches only local incident elements and returns to its start without global search |
| Scene graph | Tree of local homogeneous transforms traversed with a matrix stack | Recursive stack product matches explicit matrix multiplication and the stack is restored after traversal |
| Ray-tracing instancing | Transform the ray into object space instead of baking transformed geometry | Ray parameter is unchanged by inverse-transform intersection; normals need inverse-transpose handling |
| Bounding boxes and BVHs | Ray parameter intervals over axis-aligned slabs and a binary object hierarchy | AABB intervals overlap exactly when the ray can hit; BVH tests fewer primitives than brute force |
| Uniform spatial subdivision | Regular grid cells plus incremental ray marching | Traversed cells form a monotone path and candidate tests are restricted to visited cells |
| BSP trees | Signed plane function, side classification, visibility traversal order, and triangle splitting | Child triangles do not span the split plane; split area equals original area within tolerance |
| Tiled multidimensional arrays | Integer decomposition into tile coordinates and local offsets | Direct formula equals lookup-table formula; tiled vertical scans have smaller memory jumps than row-major scans |

A useful habit across the chapter is to separate **where data lives** from **what query it answers**. Indexed meshes are compact for drawing; half-edges are better for editing. BVHs group objects; grids partition space. Scene graphs compose coordinate systems; instancing delays transformed geometry until an intersection or draw call asks for it.

## Visual storyboard

| Artifact | Concept | Representation and library | What to inspect | Validation |
| --- | --- | --- | --- | --- |
| `figures/mesh-connectivity-storage-audit.png` | Mesh storage and connectivity | Trimesh topology checks, NetworkX one-ring traversal, Matplotlib diagram | Edge incidence counts, strip/fan savings, half-edge local circulation | Euler characteristic, watertightness, winding consistency, pair/next invariants |
| `figures/scene-graph-transform-stack.png` | Scene graph traversal | NetworkX tree plus 2D homogeneous transforms | Matrix-stack ancestry and resulting ferry/car/wheel positions | Recursive world transforms equal explicit products |
| `figures/spatial-acceleration-ray-traversal.png` and `html/spatial-acceleration-hover-audit.html` | Bounding volumes, BVH, and uniform grid | Custom AABB/BVH/grid traversal with Matplotlib and Plotly | Which boxes/cells are visited before primitive tests | Traversal uses fewer primitive tests than brute force and reports the same first hit |
| `figures/bsp-triangle-split-visibility-order.png` | BSP tree classification, splitting, and visibility order | Signed plane function and Matplotlib split diagram | Which child triangles land on each side and how viewpoint changes order | Split children conserve area and no child spans the plane |
| `figures/tiled-array-locality-scan.png` | Tiled and blocked arrays | NumPy index formulas, Matplotlib heatmaps, CSV sweep | Row-major versus tiled memory jumps for vertical and stencil-like scans | Tiled formula equals table lookup and 3D two-level blocked formula decomposes as Fx+Fy+Fz |

These are not textbook figures. They are small generated audits that expose the same implementation pressures described in the source pages: compact storage, local adjacency, hierarchical transforms, ray culling, stable plane tests, and memory locality.


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import math
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib import patches
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import trimesh

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

CHAPTER = 12
TOPIC = "chapter-12"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(12)
np.set_printoptions(precision=4, suppress=True)

source_span = {
    "book": "Fundamentals of Computer Graphics",
    "chapter": 12,
    "title": "Data Structures for Graphics",
    "printed_pages": "291-334",
    "pdf_pages": "308-351",
    "topics_read": [
        "triangle meshes and topology",
        "indexed meshes, triangle strips, and fans",
        "triangle-neighbor, winged-edge, and half-edge connectivity",
        "scene graphs and matrix stacks",
        "ray-tracing instancing",
        "bounding boxes, BVHs, uniform grids, and axis-aligned BSP trees",
        "BSP trees for visibility and triangle splitting",
        "tiled multidimensional arrays",
    ],
}
source_span_path = save_json(source_span, TOPIC, "source-span.json")

storyboard = {
    "chapter_goal": "Choose graphics data structures by the local queries and invariants they make cheap.",
    "source_span_read": source_span,
    "visual_sequence": [
        "mesh-connectivity-storage-audit.png",
        "scene-graph-transform-stack.png",
        "spatial-acceleration-ray-traversal.png",
        "spatial-acceleration-hover-audit.html",
        "bsp-triangle-split-visibility-order.png",
        "tiled-array-locality-scan.png",
    ],
    "library_routing": {
        "trimesh": "mesh topology, Euler characteristic, watertightness, and winding checks",
        "networkx": "connectivity diagrams and scene-graph traversal structure",
        "matplotlib": "durable labeled 2D diagrams and locality heatmaps",
        "plotly": "interactive hoverable spatial acceleration audit",
        "numpy/pandas": "indexing, traversal, and reproducible check tables",
    },
}
storyboard_path = save_json(storyboard, TOPIC, "visual-storyboard.json")

print(book_relative(source_span_path))
print(book_relative(storyboard_path))



## 1. Triangle meshes: compact draw data versus editable connectivity

A triangle mesh stores a surface as shared vertices and triangular faces. The compact version is an indexed mesh: one vertex array and one face-index array. That is excellent for drawing and transfer to a graphics API, but it does not directly answer editing queries such as "which faces touch this vertex?" or "what face is across this edge?"

The source chapter separates two tasks that are easy to confuse:

- **Storage for static drawing:** separate triangles, indexed meshes, triangle fans, and triangle strips reduce memory or bandwidth.
- **Connectivity for local surgery:** triangle-neighbor, winged-edge, and half-edge structures keep enough local pointers to walk faces, edges, and vertices without scanning the whole mesh.

The small mesh below is a square pyramid with a triangulated base. It is intentionally tiny, but it exercises the same checks that matter for large meshes: each closed edge has two incident faces, adjacent faces disagree on directed edge order, a half-edge has an opposite pair, and a vertex one-ring traversal returns to its start.


In [ ]:
def pyramid_mesh() -> tuple[np.ndarray, np.ndarray]:
    vertices = np.array(
        [
            [0.0, 0.0, 0.0],
            [1.0, 0.0, 0.0],
            [1.0, 1.0, 0.0],
            [0.0, 1.0, 0.0],
            [0.5, 0.5, 0.85],
        ],
        dtype=float,
    )
    faces = np.array(
        [
            [0, 2, 1],
            [0, 3, 2],
            [0, 1, 4],
            [1, 2, 4],
            [2, 3, 4],
            [3, 0, 4],
        ],
        dtype=int,
    )
    return vertices, faces


def undirected_edge_counts(faces: np.ndarray) -> Counter[tuple[int, int]]:
    counts: Counter[tuple[int, int]] = Counter()
    for face in faces:
        for a, b in [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]:
            counts[tuple(sorted((int(a), int(b))))] += 1
    return counts


def oriented_edge_counts(faces: np.ndarray) -> Counter[tuple[int, int]]:
    counts: Counter[tuple[int, int]] = Counter()
    for face in faces:
        for a, b in [(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]:
            counts[(int(a), int(b))] += 1
    return counts


def build_halfedges(faces: np.ndarray) -> list[dict[str, int]]:
    halfedges: list[dict[str, int]] = []
    directed_to_halfedge: dict[tuple[int, int], int] = {}
    for f_idx, face in enumerate(faces):
        start = len(halfedges)
        for local, (tail, head) in enumerate([(face[0], face[1]), (face[1], face[2]), (face[2], face[0])]):
            h_idx = start + local
            halfedges.append(
                {
                    "id": h_idx,
                    "face": int(f_idx),
                    "tail": int(tail),
                    "head": int(head),
                    "next": start + ((local + 1) % 3),
                    "pair": -1,
                }
            )
            directed_to_halfedge[(int(tail), int(head))] = h_idx
    for h in halfedges:
        h["pair"] = directed_to_halfedge[(h["head"], h["tail"])]
    return halfedges


def one_ring_faces_to_vertex(halfedges: list[dict[str, int]], vertex: int) -> list[int]:
    incoming = [h for h in halfedges if h["head"] == vertex]
    start = min(incoming, key=lambda h: h["face"])
    h = start
    faces_seen = []
    for _ in range(len(incoming) + 2):
        faces_seen.append(h["face"])
        h = halfedges[halfedges[halfedges[h["pair"]]["next"]]["next"]]
        if h["id"] == start["id"]:
            break
    return faces_seen


def strip_triangles(stream: list[int]) -> list[tuple[int, int, int]]:
    tris = []
    for i in range(len(stream) - 2):
        a, b, c = stream[i : i + 3]
        tris.append((a, b, c) if i % 2 == 0 else (c, b, a))
    return tris


def fan_triangles(stream: list[int]) -> list[tuple[int, int, int]]:
    return [(stream[0], stream[i], stream[i + 1]) for i in range(1, len(stream) - 1)]

vertices, faces = pyramid_mesh()
mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
edge_counts = undirected_edge_counts(faces)
halfedges = build_halfedges(faces)
one_ring_apex = one_ring_faces_to_vertex(halfedges, 4)

fan_stream = [0, 1, 2, 3, 4, 5]
strip_stream = [0, 1, 2, 3, 4, 5, 6, 7]
strip_tris = strip_triangles(strip_stream)
fan_tris = fan_triangles(fan_stream)

nv, nt = len(vertices), len(faces)
storage_units = {
    "separate triangles": 9 * nt,
    "indexed mesh": 3 * nv + 3 * nt,
    "triangle-neighbor": 4 * nv + 6 * nt,
    "half-edge rough": 3 * nv + 3 * nt + 4 * (3 * nt),
}
pair_ok = all(halfedges[h["pair"]]["pair"] == h["id"] for h in halfedges)
next_cycles_ok = all(
    halfedges[halfedges[halfedges[h["next"]]["next"]]["next"]]["id"] == h["id"]
    for h in halfedges
)
oriented = oriented_edge_counts(faces)
winding_pairs_ok = all(oriented[(b, a)] == oriented[(a, b)] for a, b in oriented if a < b)

mesh_checks = {
    "vertices": int(nv),
    "faces": int(nt),
    "undirected_edges": int(len(edge_counts)),
    "euler_characteristic": int(nv - len(edge_counts) + nt),
    "edge_incidence_counts": sorted(edge_counts.values()),
    "closed_edge_count": int(sum(c == 2 for c in edge_counts.values())),
    "boundary_edge_count": int(sum(c == 1 for c in edge_counts.values())),
    "nonmanifold_edge_count": int(sum(c > 2 for c in edge_counts.values())),
    "trimesh_watertight": bool(mesh.is_watertight),
    "trimesh_winding_consistent": bool(mesh.is_winding_consistent),
    "trimesh_euler_number": int(mesh.euler_number),
    "halfedge_pair_reciprocal": bool(pair_ok),
    "halfedge_face_next_cycles": bool(next_cycles_ok),
    "opposite_directed_edges_match": bool(winding_pairs_ok),
    "apex_one_ring_faces": [int(x) for x in one_ring_apex],
    "apex_valence": int(len(one_ring_apex)),
    "strip_triangles_from_8_indices": len(strip_tris),
    "fan_triangles_from_6_indices": len(fan_tris),
    "strip_index_units_for_6_triangles": len(strip_stream),
    "indexed_index_units_for_6_triangles": 3 * len(strip_tris),
    "strip_relative_index_size": float(len(strip_stream) / (3 * len(strip_tris))),
    "storage_units": storage_units,
}
mesh_check_path = save_json(mesh_checks, TOPIC, "mesh-connectivity-checks.json")
mesh_checks



In [ ]:
fig = plt.figure(figsize=(13.5, 4.8))
grid = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.0, 1.1])

ax = fig.add_subplot(grid[0, 0], projection="3d")
for face_id, face in enumerate(faces):
    poly = vertices[face]
    loop = np.vstack([poly, poly[0]])
    ax.plot(loop[:, 0], loop[:, 1], loop[:, 2], color=PALETTE["blue"], linewidth=2)
    center = poly.mean(axis=0)
    ax.text(center[0], center[1], center[2] + 0.035, f"T{face_id}", fontsize=8, color=PALETTE["ink"])
for idx, v in enumerate(vertices):
    ax.scatter(v[0], v[1], v[2], color=PALETTE["red"] if idx == 4 else PALETTE["gold"], s=45)
    ax.text(v[0], v[1], v[2] + 0.06, f"v{idx}", fontsize=8)
ax.set_title("indexed mesh: shared vertices and closed edges", fontsize=10)
ax.set_box_aspect((1, 1, 0.8))
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=24, azim=-54)

ax2 = fig.add_subplot(grid[0, 1])
labels = list(storage_units)
values = [storage_units[k] for k in labels]
colors = [PALETTE["gray"], PALETTE["blue"], PALETTE["teal"], PALETTE["violet"]]
ax2.barh(labels, values, color=colors)
for label, value in zip(labels, values):
    ax2.text(value + 1.0, label, str(value), va="center", fontsize=8)
style_axis(ax2, "storage units for this 6-triangle mesh", xlabel="float/int/pointer-sized units")
ax2.set_xlim(0, max(values) * 1.18)

ax3 = fig.add_subplot(grid[0, 2])
G = nx.DiGraph()
for face_id in one_ring_apex:
    G.add_node(f"T{face_id}")
ring_nodes = [f"T{face_id}" for face_id in one_ring_apex]
for a, b in zip(ring_nodes, ring_nodes[1:] + ring_nodes[:1]):
    G.add_edge(a, b)
pos = nx.circular_layout(G)
nx.draw_networkx_nodes(G, pos, node_color=PALETTE["teal"], node_size=1000, ax=ax3)
nx.draw_networkx_labels(G, pos, font_color="white", font_size=9, ax=ax3)
nx.draw_networkx_edges(G, pos, edge_color=PALETTE["ink"], arrows=True, arrowstyle="-|>", width=1.8, ax=ax3)
ax3.text(0.0, 0.0, "v4\napex", ha="center", va="center", fontsize=10, color=PALETTE["red"], fontweight="bold")
ax3.set_title("half-edge one-ring traversal: pair.next.next", fontsize=10)
ax3.axis("off")

fig.suptitle("Mesh data structures: compact arrays plus local connectivity", fontsize=13, color=PALETTE["ink"])
mesh_fig_path = save_matplotlib(fig, TOPIC, "mesh-connectivity-storage-audit.png")
close(fig)
display_artifact(mesh_fig_path, width=900)
display_artifact(mesh_check_path)



The bars show why the chapter treats indexed meshes and editable connectivity structures separately. The indexed mesh is smaller than separate triangles because positions are shared. The half-edge style representation is larger, but it buys local traversal: following `pair.next.next` around `v4` in this local orientation visits only the incident side faces and then returns to the start. That is the constant-time-per-neighbor behavior the source emphasizes.

Triangle strips and fans are another storage idea, not an editing structure. A strip with eight indices encodes six triangles, while a conventional indexed mesh would need eighteen face indices for the same six triangles. The parity flip in the strip construction keeps adjacent triangles consistently oriented.


## 2. Scene graphs: transformations live on paths

A scene graph stores objects in local coordinates and composes transforms along the path from the root. This is more than a convenient drawing tree: it keeps relative motion local. Moving the car on a ferry should move the wheels with the car. Rotating a wheel should not move the ferry.

The code below uses a 2D homogeneous-coordinate graph so the matrix products are readable. The traversal mimics a matrix stack: push a local transform when entering a node, draw with the product, recursively visit children, then pop back to the parent state.



In [ ]:
def translate2(tx: float, ty: float) -> np.ndarray:
    return np.array([[1.0, 0.0, tx], [0.0, 1.0, ty], [0.0, 0.0, 1.0]])


def rotate2(theta: float) -> np.ndarray:
    c, s = math.cos(theta), math.sin(theta)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])


def scale2(sx: float, sy: float) -> np.ndarray:
    return np.array([[sx, 0.0, 0.0], [0.0, sy, 0.0], [0.0, 0.0, 1.0]])


def apply2(M: np.ndarray, pts: np.ndarray) -> np.ndarray:
    h = np.column_stack([pts, np.ones(len(pts))])
    return (M @ h.T).T[:, :2]

@dataclass
class SceneNode:
    name: str
    parent: str | None
    local: np.ndarray
    kind: str

nodes = {
    "world": SceneNode("world", None, np.eye(3), "root"),
    "ferry": SceneNode("ferry", "world", translate2(0.4, 0.2) @ rotate2(math.radians(8)), "body"),
    "car": SceneNode("car", "ferry", translate2(1.05, 0.18) @ rotate2(math.radians(-5)), "body"),
    "front-left-wheel": SceneNode("front-left-wheel", "car", translate2(0.48, 0.27) @ rotate2(math.radians(35)), "wheel"),
    "front-right-wheel": SceneNode("front-right-wheel", "car", translate2(0.48, -0.27) @ rotate2(math.radians(35)), "wheel"),
    "rear-left-wheel": SceneNode("rear-left-wheel", "car", translate2(-0.48, 0.27) @ rotate2(math.radians(35)), "wheel"),
    "rear-right-wheel": SceneNode("rear-right-wheel", "car", translate2(-0.48, -0.27) @ rotate2(math.radians(35)), "wheel"),
    "mast": SceneNode("mast", "ferry", translate2(-1.0, 0.0), "mast"),
}
children: dict[str, list[str]] = defaultdict(list)
for name, node in nodes.items():
    if node.parent is not None:
        children[node.parent].append(name)

world_transforms: dict[str, np.ndarray] = {}
stack_depths: dict[str, int] = {}
max_stack_depth = 0

def traverse_scene(name: str, stack: list[np.ndarray]) -> None:
    global max_stack_depth
    composite = stack[-1] @ nodes[name].local
    stack.append(composite)
    world_transforms[name] = composite
    stack_depths[name] = len(stack) - 1
    max_stack_depth = max(max_stack_depth, len(stack) - 1)
    for child in children[name]:
        traverse_scene(child, stack)
    stack.pop()

stack = [np.eye(3)]
traverse_scene("world", stack)
stack_restored = len(stack) == 1 and np.allclose(stack[0], np.eye(3))
explicit_front = nodes["world"].local @ nodes["ferry"].local @ nodes["car"].local @ nodes["front-left-wheel"].local
explicit_match = np.allclose(world_transforms["front-left-wheel"], explicit_front)

circle = np.column_stack([np.cos(np.linspace(0, 2 * np.pi, 60)), np.sin(np.linspace(0, 2 * np.pi, 60))])
ferry_shape = np.array([[-1.7, -0.45], [1.7, -0.45], [1.9, 0.0], [1.7, 0.45], [-1.7, 0.45], [-1.9, 0.0], [-1.7, -0.45]])
car_shape = np.array([[-0.72, -0.32], [0.72, -0.32], [0.72, 0.32], [-0.72, 0.32], [-0.72, -0.32]])
mast_shape = np.array([[0.0, -0.05], [0.0, 0.75]])

scene_checks = {
    "node_count": len(nodes),
    "edge_count": sum(len(v) for v in children.values()),
    "max_stack_depth": int(max_stack_depth),
    "stack_restored_after_traversal": bool(stack_restored),
    "front_left_wheel_explicit_product_match": bool(explicit_match),
    "front_left_wheel_world_center": apply2(world_transforms["front-left-wheel"], np.array([[0.0, 0.0]])).round(6).tolist()[0],
}
scene_check_path = save_json(scene_checks, TOPIC, "scene-graph-transform-checks.json")
scene_checks



In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13.2, 5.0), gridspec_kw={"width_ratios": [0.95, 1.15]})

SG = nx.DiGraph()
for name, node in nodes.items():
    SG.add_node(name, kind=node.kind)
    if node.parent:
        SG.add_edge(node.parent, name)
try:
    pos = nx.nx_agraph.graphviz_layout(SG, prog="dot")
except Exception:
    pos = nx.spring_layout(SG, seed=12)
node_colors = [PALETTE["gold"] if n == "world" else PALETTE["blue"] if nodes[n].kind == "body" else PALETTE["teal"] for n in SG.nodes]
nx.draw_networkx_edges(SG, pos, ax=ax0, arrows=True, edge_color="#9aa6b2", arrowstyle="-|>")
nx.draw_networkx_nodes(SG, pos, ax=ax0, node_color=node_colors, node_size=1050)
nx.draw_networkx_labels(SG, pos, ax=ax0, font_size=7, font_color="white")
edge_labels = {(u, v): f"depth {stack_depths[v]}" for u, v in SG.edges}
nx.draw_networkx_edge_labels(SG, pos, edge_labels=edge_labels, ax=ax0, font_size=7)
ax0.set_title("scene graph: local transforms compose along paths")
ax0.axis("off")

ferry_world = apply2(world_transforms["ferry"], ferry_shape)
car_world = apply2(world_transforms["car"], car_shape)
ax1.plot(ferry_world[:, 0], ferry_world[:, 1], color=PALETTE["blue"], linewidth=2.5, label="ferry")
ax1.plot(car_world[:, 0], car_world[:, 1], color=PALETTE["red"], linewidth=2.2, label="car")
for wheel_name in ["front-left-wheel", "front-right-wheel", "rear-left-wheel", "rear-right-wheel"]:
    wheel_pts = apply2(world_transforms[wheel_name] @ scale2(0.12, 0.12), circle)
    ax1.plot(wheel_pts[:, 0], wheel_pts[:, 1], color=PALETTE["ink"], linewidth=1.5)
    center = apply2(world_transforms[wheel_name], np.array([[0.0, 0.0]]))[0]
    ax1.scatter(center[0], center[1], color=PALETTE["gold"], s=25)
mast_world = apply2(world_transforms["mast"], mast_shape)
ax1.plot(mast_world[:, 0], mast_world[:, 1], color=PALETTE["teal"], linewidth=3, label="mast")

origin_frame = apply2(world_transforms["car"], np.array([[0.0, 0.0]]))[0]
x_axis = apply2(world_transforms["car"], np.array([[0.45, 0.0]]))[0]
y_axis = apply2(world_transforms["car"], np.array([[0.0, 0.45]]))[0]
ax1.arrow(origin_frame[0], origin_frame[1], x_axis[0] - origin_frame[0], x_axis[1] - origin_frame[1], color=PALETTE["red"], head_width=0.04, length_includes_head=True)
ax1.arrow(origin_frame[0], origin_frame[1], y_axis[0] - origin_frame[0], y_axis[1] - origin_frame[1], color=PALETTE["teal"], head_width=0.04, length_includes_head=True)
ax1.text(origin_frame[0], origin_frame[1] + 0.12, "car local frame", fontsize=8, color=PALETTE["ink"])
style_axis(ax1, "world-space result of the matrix stack", equal=True, xlabel="world x", ylabel="world y")
ax1.legend(loc="upper right", fontsize=8)

scene_fig_path = save_matplotlib(fig, TOPIC, "scene-graph-transform-stack.png")
close(fig)
display_artifact(scene_fig_path, width=900)
display_artifact(scene_check_path)



The scene graph figure should be read from left to right. The left panel is the data structure; the right panel is the world-space consequence of multiplying along each root-to-node path. The check for the front-left wheel compares the recursive traversal result with the explicit product `world * ferry * car * wheel`. If those ever differ, either traversal order or stack restoration is wrong.

Ray tracing uses the same transform ownership idea differently: a transformed object can keep its simple local representation, and the ray is inverse-transformed into that local space for intersection. That is why ray directions should not be forced to unit length throughout a renderer: the inverse-transformed direction may have a different length, while the ray parameter `t` still indexes the same point along the transformed ray.


## 3. Spatial data structures: cull with intervals before testing primitives

The chapter's spatial structures answer the same question in several ways: given a ray, which objects are even worth testing?

- A **bounding box** turns a 2D or 3D hit test into overlapping intervals of the ray parameter `t`.
- A **BVH** partitions objects into a hierarchy. Sibling boxes may overlap in space, but each object belongs to one child group.
- A **uniform grid** partitions space. Each point belongs to one cell, but an object can appear in several cells.
- An **axis-aligned BSP** or kd-style tree partitions space recursively with planes; objects crossing a plane may be duplicated.

The synthetic scene below uses discs because their exact ray intersection is short and auditable. The acceleration structures still store only bounding boxes until a primitive test is necessary.



In [ ]:
@dataclass(frozen=True)
class Disc:
    name: str
    center: np.ndarray
    radius: float

    @property
    def bmin(self) -> np.ndarray:
        return self.center - self.radius

    @property
    def bmax(self) -> np.ndarray:
        return self.center + self.radius


discs = [
    Disc("A", np.array([0.9, 1.0]), 0.28),
    Disc("B", np.array([1.8, 3.9]), 0.38),
    Disc("C", np.array([2.7, 2.2]), 0.35),
    Disc("D", np.array([3.5, 4.7]), 0.45),
    Disc("E", np.array([4.2, 1.35]), 0.42),
    Disc("F", np.array([5.4, 3.0]), 0.48),
    Disc("G", np.array([6.2, 1.0]), 0.36),
    Disc("H", np.array([7.0, 4.2]), 0.42),
    Disc("I", np.array([8.2, 2.2]), 0.52),
    Disc("J", np.array([9.0, 5.1]), 0.35),
]

def union_bounds(indices: list[int]) -> tuple[np.ndarray, np.ndarray]:
    bmin = np.min([discs[i].bmin for i in indices], axis=0)
    bmax = np.max([discs[i].bmax for i in indices], axis=0)
    return bmin, bmax


def ray_aabb_interval(origin: np.ndarray, direction: np.ndarray, bmin: np.ndarray, bmax: np.ndarray, t0: float = 0.0, t1: float = np.inf) -> tuple[bool, float, float]:
    with np.errstate(divide="ignore", invalid="ignore"):
        inv = 1.0 / direction
        lo = (bmin - origin) * inv
        hi = (bmax - origin) * inv
    near = np.minimum(lo, hi)
    far = np.maximum(lo, hi)
    enter = max(t0, float(np.nanmax(near)))
    exit = min(t1, float(np.nanmin(far)))
    return bool(enter <= exit), enter, exit


def ray_disc_hit(origin: np.ndarray, direction: np.ndarray, disc: Disc, t0: float = 0.0, t1: float = np.inf) -> float | None:
    oc = origin - disc.center
    a = float(direction @ direction)
    b = float(2.0 * (oc @ direction))
    c = float(oc @ oc - disc.radius * disc.radius)
    discr = b * b - 4 * a * c
    if discr < 0:
        return None
    root = math.sqrt(discr)
    candidates = [(-b - root) / (2 * a), (-b + root) / (2 * a)]
    valid = [t for t in candidates if t0 <= t <= t1]
    return min(valid) if valid else None

@dataclass
class BVHNode:
    indices: list[int]
    bmin: np.ndarray
    bmax: np.ndarray
    left: "BVHNode | None" = None
    right: "BVHNode | None" = None
    depth: int = 0

    @property
    def is_leaf(self) -> bool:
        return self.left is None and self.right is None


def build_bvh(indices: list[int], depth: int = 0) -> BVHNode:
    bmin, bmax = union_bounds(indices)
    if len(indices) <= 2:
        return BVHNode(indices=indices, bmin=bmin, bmax=bmax, depth=depth)
    centers = np.array([discs[i].center for i in indices])
    axis = int(np.argmax(centers.max(axis=0) - centers.min(axis=0)))
    ordered = sorted(indices, key=lambda i: discs[i].center[axis])
    mid = len(ordered) // 2
    return BVHNode(
        indices=indices,
        bmin=bmin,
        bmax=bmax,
        left=build_bvh(ordered[:mid], depth + 1),
        right=build_bvh(ordered[mid:], depth + 1),
        depth=depth,
    )


def flatten_bvh(node: BVHNode) -> list[BVHNode]:
    out = [node]
    if node.left is not None:
        out.extend(flatten_bvh(node.left))
    if node.right is not None:
        out.extend(flatten_bvh(node.right))
    return out


def traverse_bvh(node: BVHNode, origin: np.ndarray, direction: np.ndarray) -> dict[str, object]:
    visited_nodes: list[BVHNode] = []
    primitive_tests: list[str] = []
    hits: list[tuple[float, str]] = []

    def visit(n: BVHNode, t0: float, t1: float) -> None:
        hit_box, enter, exit = ray_aabb_interval(origin, direction, n.bmin, n.bmax, t0, t1)
        if not hit_box:
            return
        visited_nodes.append(n)
        if n.is_leaf:
            for idx in n.indices:
                primitive_tests.append(discs[idx].name)
                t = ray_disc_hit(origin, direction, discs[idx], t0, t1)
                if t is not None:
                    hits.append((t, discs[idx].name))
        else:
            children_here = [c for c in [n.left, n.right] if c is not None]
            child_entries = []
            for child in children_here:
                h, e, _ = ray_aabb_interval(origin, direction, child.bmin, child.bmax, t0, t1)
                if h:
                    child_entries.append((e, child))
            for _, child in sorted(child_entries, key=lambda item: item[0]):
                visit(child, t0, t1)

    visit(node, 0.0, np.inf)
    hits_sorted = sorted(hits)
    return {"visited_nodes": visited_nodes, "primitive_tests": primitive_tests, "hits": hits_sorted, "first_hit": hits_sorted[0] if hits_sorted else None}

origin = np.array([-0.65, 0.72])
direction = np.array([1.0, 0.25])
direction = direction / np.linalg.norm(direction)
root = build_bvh(list(range(len(discs))))
bvh_result = traverse_bvh(root, origin, direction)
brute_hits = sorted((t, disc.name) for disc in discs if (t := ray_disc_hit(origin, direction, disc)) is not None)
vertical_box_hit = ray_aabb_interval(np.array([0.5, -1.0]), np.array([0.0, 1.0]), np.array([0.0, 0.0]), np.array([1.0, 1.0]))
miss_box_hit = ray_aabb_interval(np.array([-1.0, 2.0]), np.array([0.0, 1.0]), np.array([0.0, 0.0]), np.array([1.0, 1.0]))

bvh_result["first_hit"], brute_hits[:2], [d for d in bvh_result["primitive_tests"]]


In [ ]:
grid_bmin = np.array([0.0, 0.0])
grid_bmax = np.array([10.0, 6.0])
grid_shape = np.array([5, 3])
cell_size = (grid_bmax - grid_bmin) / grid_shape

def cells_for_disc(disc: Disc) -> list[tuple[int, int]]:
    lo = np.floor((disc.bmin - grid_bmin) / cell_size).astype(int)
    hi = np.floor((disc.bmax - grid_bmin) / cell_size).astype(int)
    lo = np.clip(lo, 0, grid_shape - 1)
    hi = np.clip(hi, 0, grid_shape - 1)
    return [(i, j) for i in range(lo[0], hi[0] + 1) for j in range(lo[1], hi[1] + 1)]

grid_cells: dict[tuple[int, int], list[int]] = defaultdict(list)
for idx, disc in enumerate(discs):
    for cell in cells_for_disc(disc):
        grid_cells[cell].append(idx)


def ray_grid_cells(origin: np.ndarray, direction: np.ndarray) -> list[tuple[int, int]]:
    hit, t_enter, t_exit = ray_aabb_interval(origin, direction, grid_bmin, grid_bmax)
    if not hit:
        return []
    t = max(t_enter, 0.0) + 1e-9
    p = origin + t * direction
    cell = np.floor((p - grid_bmin) / cell_size).astype(int)
    cell = np.clip(cell, 0, grid_shape - 1)
    step = np.sign(direction).astype(int)
    step[step == 0] = 1
    next_boundary = grid_bmin + (cell + (step > 0)) * cell_size
    with np.errstate(divide="ignore", invalid="ignore"):
        t_max = np.where(direction != 0, (next_boundary - origin) / direction, np.inf)
        t_delta = np.where(direction != 0, cell_size / np.abs(direction), np.inf)
    visited = []
    while 0 <= cell[0] < grid_shape[0] and 0 <= cell[1] < grid_shape[1] and t <= t_exit + 1e-9:
        visited.append((int(cell[0]), int(cell[1])))
        if t_max[0] < t_max[1]:
            cell[0] += step[0]
            t = t_max[0]
            t_max[0] += t_delta[0]
        else:
            cell[1] += step[1]
            t = t_max[1]
            t_max[1] += t_delta[1]
    return visited

visited_cells = ray_grid_cells(origin, direction)
grid_candidates_ordered = []
seen_candidates = set()
for cell in visited_cells:
    for idx in grid_cells[cell]:
        if idx not in seen_candidates:
            seen_candidates.add(idx)
            grid_candidates_ordered.append(discs[idx].name)

grid_hits = []
for name in grid_candidates_ordered:
    disc = next(d for d in discs if d.name == name)
    t = ray_disc_hit(origin, direction, disc)
    if t is not None:
        grid_hits.append((t, name))
grid_hits = sorted(grid_hits)

traversal_rows = []
for order, cell in enumerate(visited_cells):
    traversal_rows.append(
        {
            "visit_order": order,
            "cell_x": cell[0],
            "cell_y": cell[1],
            "object_names": " ".join(discs[i].name for i in grid_cells[cell]),
        }
    )
traversal_table_path = save_table_csv(traversal_rows, TOPIC, "spatial-grid-traversal-events.csv")

spatial_checks = {
    "object_count": len(discs),
    "brute_force_primitive_tests": len(discs),
    "bvh_visited_node_count": len(bvh_result["visited_nodes"]),
    "bvh_primitive_tests": len(bvh_result["primitive_tests"]),
    "grid_visited_cell_count": len(visited_cells),
    "grid_candidate_tests": len(grid_candidates_ordered),
    "first_hit_brute_force": brute_hits[0] if brute_hits else None,
    "first_hit_bvh": bvh_result["first_hit"],
    "first_hit_grid": grid_hits[0] if grid_hits else None,
    "vertical_inside_slab_hits": bool(vertical_box_hit[0]),
    "parallel_outside_slab_misses": bool(not miss_box_hit[0]),
    "visited_cells": visited_cells,
    "bvh_primitive_test_names": bvh_result["primitive_tests"],
    "grid_candidate_names": grid_candidates_ordered,
}
spatial_check_path = save_json(spatial_checks, TOPIC, "spatial-traversal-checks.json")
spatial_checks



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.8), gridspec_kw={"width_ratios": [1.25, 1.15, 0.8]})
ax = axes[0]
for node in flatten_bvh(root):
    color = PALETTE["red"] if node in bvh_result["visited_nodes"] else "#c8d0da"
    lw = 2.0 if node in bvh_result["visited_nodes"] else 0.9
    rect = patches.Rectangle(node.bmin, *(node.bmax - node.bmin), fill=False, edgecolor=color, linewidth=lw, alpha=0.9)
    ax.add_patch(rect)
for disc in discs:
    c = patches.Circle(disc.center, disc.radius, facecolor="#dbeafe", edgecolor=PALETTE["blue"], alpha=0.95)
    ax.add_patch(c)
    ax.text(disc.center[0], disc.center[1], disc.name, ha="center", va="center", fontsize=8, color=PALETTE["ink"])
ray_end = origin + 12 * direction
ax.plot([origin[0], ray_end[0]], [origin[1], ray_end[1]], color=PALETTE["red"], linewidth=2.2, label="ray")
style_axis(ax, "BVH: visited boxes before primitive tests", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(-1, 10.5)
ax.set_ylim(-0.2, 6.3)

ax = axes[1]
for i in range(grid_shape[0]):
    for j in range(grid_shape[1]):
        xy = grid_bmin + np.array([i, j]) * cell_size
        face = "#fff7d6" if (i, j) in visited_cells else "white"
        rect = patches.Rectangle(xy, cell_size[0], cell_size[1], facecolor=face, edgecolor="#b6c0ca", linewidth=1.0)
        ax.add_patch(rect)
        if grid_cells[(i, j)]:
            ax.text(xy[0] + 0.08, xy[1] + 0.12, "".join(discs[k].name for k in grid_cells[(i, j)]), fontsize=7, color=PALETTE["gray"])
for order, (i, j) in enumerate(visited_cells):
    xy = grid_bmin + np.array([i + 0.5, j + 0.5]) * cell_size
    ax.text(xy[0], xy[1], str(order), ha="center", va="center", fontsize=9, color=PALETTE["red"], fontweight="bold")
for disc in discs:
    ax.add_patch(patches.Circle(disc.center, disc.radius, facecolor="none", edgecolor=PALETTE["blue"], linewidth=1.2))
ax.plot([origin[0], ray_end[0]], [origin[1], ray_end[1]], color=PALETTE["red"], linewidth=2.2)
style_axis(ax, "uniform grid: DDA visits cells in order", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(-1, 10.5)
ax.set_ylim(-0.2, 6.3)

ax = axes[2]
labels = ["brute", "BVH", "grid"]
values = [spatial_checks["brute_force_primitive_tests"], spatial_checks["bvh_primitive_tests"], spatial_checks["grid_candidate_tests"]]
ax.bar(labels, values, color=[PALETTE["gray"], PALETTE["teal"], PALETTE["gold"]])
for label, value in zip(labels, values):
    ax.text(label, value + 0.2, str(value), ha="center", fontsize=9)
style_axis(ax, "primitive candidate tests", ylabel="count")
ax.set_ylim(0, max(values) + 2)

fig.suptitle("Spatial data structures trade preprocessing for fewer ray-object tests", fontsize=13, color=PALETTE["ink"])
spatial_fig_path = save_matplotlib(fig, TOPIC, "spatial-acceleration-ray-traversal.png")
close(fig)
display_artifact(spatial_fig_path, width=950)
display_artifact(spatial_check_path)
display_artifact(traversal_table_path)



In [ ]:
plotly_fig = go.Figure()
angles = np.linspace(0, 2 * np.pi, 80)
for disc in discs:
    x = disc.center[0] + disc.radius * np.cos(angles)
    y = disc.center[1] + disc.radius * np.sin(angles)
    tested = disc.name in spatial_checks["bvh_primitive_test_names"] or disc.name in spatial_checks["grid_candidate_names"]
    plotly_fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            name=f"disc {disc.name}",
            line={"color": PALETTE["red"] if tested else PALETTE["blue"], "width": 2},
            hovertemplate=f"disc {disc.name}<br>center=({disc.center[0]:.2f},{disc.center[1]:.2f})<br>radius={disc.radius:.2f}<extra></extra>",
        )
    )
for node_id, node in enumerate(flatten_bvh(root)):
    if node not in bvh_result["visited_nodes"]:
        continue
    x0, y0 = node.bmin
    x1, y1 = node.bmax
    plotly_fig.add_trace(
        go.Scatter(
            x=[x0, x1, x1, x0, x0],
            y=[y0, y0, y1, y1, y0],
            mode="lines",
            name=f"visited BVH box {node_id}",
            line={"color": "rgba(196,78,82,0.55)", "width": 1},
            hovertemplate=f"BVH depth {node.depth}<br>objects={','.join(discs[i].name for i in node.indices)}<extra></extra>",
        )
    )
plotly_fig.add_trace(
    go.Scatter(
        x=[origin[0], ray_end[0]],
        y=[origin[1], ray_end[1]],
        mode="lines+markers",
        name="ray",
        line={"color": PALETTE["ink"], "width": 3},
        hovertemplate="ray traversal<extra></extra>",
    )
)
plotly_fig.update_layout(
    title="Chapter 12 spatial acceleration hover audit",
    xaxis={"scaleanchor": "y", "title": "x"},
    yaxis={"title": "y"},
    width=850,
    height=540,
    showlegend=False,
    template="plotly_white",
)
spatial_html_path = save_plotly_html(plotly_fig, TOPIC, "spatial-acceleration-hover-audit.html")
display_artifact(spatial_html_path, width="100%", height=560)



The BVH and grid both find the same first hit as the brute-force loop, but they restrict expensive primitive tests. Notice the conceptual difference in the figures: BVH boxes overlap because they group objects, while grid cells do not overlap because they partition space. The grid table is part of the lesson because it exposes the actual cell order produced by the incremental traversal.

The AABB check also includes two parallel-ray cases. A vertical ray whose x coordinate lies inside the box's x slab hits; a parallel ray outside the slab misses. That is the interval-overlap idea behind the slab test.


## 4. BSP trees for visibility: signed planes, splits, and viewpoint order

A BSP tree uses a signed plane function `f(p)` to divide primitives. For visibility sorting, the splitting plane is often the plane of a polygon rather than an axis-aligned grid plane. Traversal order depends on the viewer's sign: draw the far side, then the root polygon, then the near side.

The one hard case is a primitive that spans the plane. It must be cut into pieces that lie on one side or the other. The source chapter stresses two details that the notebook checks directly: preserve orientation when building child triangles, and use an epsilon band so a vertex extremely close to the plane does not create sliver triangles.



In [ ]:
def signed_line(p: np.ndarray, normal: np.ndarray, d: float) -> float:
    return float(normal @ p + d)


def polygon_area(poly: np.ndarray) -> float:
    x = poly[:, 0]
    y = poly[:, 1]
    return 0.5 * float(np.sum(x * np.roll(y, -1) - y * np.roll(x, -1)))


def split_triangle_by_line(tri: np.ndarray, normal: np.ndarray, d: float, eps: float = 1e-10) -> dict[str, object]:
    signed = np.array([signed_line(p, normal, d) for p in tri])
    snapped = signed.copy()
    snapped[np.abs(snapped) < eps] = 0.0
    pos_indices = [i for i, s in enumerate(snapped) if s >= 0]
    neg_indices = [i for i, s in enumerate(snapped) if s <= 0]
    if len(pos_indices) == 3:
        return {"positive": [tri], "negative": [], "signed": snapped}
    if len(neg_indices) == 3:
        return {"positive": [], "negative": [tri], "signed": snapped}

    strictly_pos = [i for i, s in enumerate(snapped) if s > 0]
    strictly_neg = [i for i, s in enumerate(snapped) if s < 0]
    if len(strictly_pos) == 1:
        c_idx = strictly_pos[0]
        a_idx, b_idx = strictly_neg
        c_side = "positive"
    else:
        c_idx = strictly_neg[0]
        a_idx, b_idx = strictly_pos
        c_side = "negative"
    a, b, c = tri[a_idx], tri[b_idx], tri[c_idx]
    fa, fb, fc = snapped[a_idx], snapped[b_idx], snapped[c_idx]

    def intersect(p0, f0, p1, f1):
        t = -f0 / (f1 - f0)
        return p0 + t * (p1 - p0)

    A = intersect(a, fa, c, fc)
    B = intersect(b, fb, c, fc)
    same_side_triangles = [np.array([a, b, A]), np.array([b, B, A])]
    c_side_triangle = np.array([A, B, c])
    if c_side == "positive":
        positive = [c_side_triangle]
        negative = same_side_triangles
    else:
        positive = same_side_triangles
        negative = [c_side_triangle]
    parent_sign = math.copysign(1.0, polygon_area(tri))
    for group in [positive, negative]:
        for k, child in enumerate(group):
            if abs(polygon_area(child)) > eps and math.copysign(1.0, polygon_area(child)) != parent_sign:
                group[k] = child[[0, 2, 1]]
    return {"positive": positive, "negative": negative, "signed": snapped, "intersection_points": [A, B]}

tri = np.array([[0.2, 0.65], [2.95, 0.25], [1.55, 2.65]])
normal = np.array([1.0, -0.28])
normal = normal / np.linalg.norm(normal)
d = -1.38
split = split_triangle_by_line(tri, normal, d, eps=1e-8)
children_split = split["positive"] + split["negative"]
parent_area = abs(polygon_area(tri))
child_areas = [abs(polygon_area(child)) for child in children_split]
area_error = abs(sum(child_areas) - parent_area)
children_span_plane = any(
    (np.max([signed_line(p, normal, d) for p in child]) > 1e-8 and np.min([signed_line(p, normal, d) for p in child]) < -1e-8)
    for child in children_split
)

near_plane_tri = np.array([[0.2, 0.1], [0.9, 0.1], [1.0e-11, 0.9]])
near_split = split_triangle_by_line(near_plane_tri, np.array([1.0, 0.0]), 0.0, eps=1e-8)
near_split_child_count = len(near_split["positive"]) + len(near_split["negative"])

eye_negative = np.array([0.2, 2.7])
eye_positive = np.array([2.8, 0.2])
visibility_orders = {}
for name, eye in {"eye_negative_side": eye_negative, "eye_positive_side": eye_positive}.items():
    visibility_orders[name] = ["positive subtree", "root triangle", "negative subtree"] if signed_line(eye, normal, d) < 0 else ["negative subtree", "root triangle", "positive subtree"]

bsp_checks = {
    "parent_area": parent_area,
    "child_area_sum": float(sum(child_areas)),
    "area_error": float(area_error),
    "positive_child_count": len(split["positive"]),
    "negative_child_count": len(split["negative"]),
    "children_span_plane": bool(children_span_plane),
    "near_plane_epsilon_child_count": int(near_split_child_count),
    "visibility_orders": visibility_orders,
}
bsp_check_path = save_json(bsp_checks, TOPIC, "bsp-split-checks.json")
bsp_checks



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.6), gridspec_kw={"width_ratios": [1.0, 1.05, 1.0]})
line_x = np.linspace(-0.1, 3.2, 100)
line_y = -(normal[0] * line_x + d) / normal[1]

ax = axes[0]
ax.fill(tri[:, 0], tri[:, 1], facecolor="#dbeafe", edgecolor=PALETTE["blue"], alpha=0.75)
ax.plot(line_x, line_y, color=PALETTE["red"], linewidth=2, label="split plane")
for i, p in enumerate(tri):
    s = signed_line(p, normal, d)
    ax.scatter(p[0], p[1], color=PALETTE["gold"] if s >= 0 else PALETTE["teal"], s=45)
    ax.text(p[0] + 0.03, p[1] + 0.03, f"v{i}: f={s:+.2f}", fontsize=8)
style_axis(ax, "before split: one triangle spans the plane", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(0, 3.2)
ax.set_ylim(0, 2.9)

ax = axes[1]
for child in split["positive"]:
    ax.fill(child[:, 0], child[:, 1], facecolor="#fde68a", edgecolor=PALETTE["gold"], alpha=0.78)
for child in split["negative"]:
    ax.fill(child[:, 0], child[:, 1], facecolor="#ccfbf1", edgecolor=PALETTE["teal"], alpha=0.78)
for pt in split["intersection_points"]:
    ax.scatter(pt[0], pt[1], color=PALETTE["red"], s=35)
    ax.text(pt[0] + 0.03, pt[1] + 0.03, "cut", fontsize=8, color=PALETTE["red"])
ax.plot(line_x, line_y, color=PALETTE["red"], linewidth=2)
style_axis(ax, "after split: children belong to one side", equal=True, xlabel="x", ylabel="y")
ax.set_xlim(0, 3.2)
ax.set_ylim(0, 2.9)

ax = axes[2]
order_labels = ["eye negative", "eye positive"]
orders = [visibility_orders["eye_negative_side"], visibility_orders["eye_positive_side"]]
ax.axis("off")
ax.set_title("BSP visibility traversal changes with viewpoint")
for row, (label, order) in enumerate(zip(order_labels, orders)):
    y = 0.78 - row * 0.34
    ax.text(0.02, y, label, fontsize=10, fontweight="bold", color=PALETTE["ink"], transform=ax.transAxes)
    for col, item in enumerate(order):
        x = 0.05 + col * 0.31
        rect = patches.FancyBboxPatch((x, y - 0.18), 0.25, 0.12, boxstyle="round,pad=0.02", facecolor=[PALETTE["teal"], PALETTE["blue"], PALETTE["gold"]][col], edgecolor="none", transform=ax.transAxes)
        ax.add_patch(rect)
        ax.text(x + 0.125, y - 0.12, item.replace(" subtree", ""), ha="center", va="center", fontsize=8, color="white", transform=ax.transAxes)
        if col < 2:
            ax.annotate("", xy=(x + 0.30, y - 0.12), xytext=(x + 0.26, y - 0.12), arrowprops={"arrowstyle": "->", "color": PALETTE["ink"]}, xycoords=ax.transAxes)
ax.text(0.02, 0.12, f"area error = {area_error:.2e}\nchildren span plane? {children_span_plane}", fontsize=9, color=PALETTE["ink"], transform=ax.transAxes)

fig.suptitle("BSP preprocessing replaces ambiguous ordering with signed-side pieces", fontsize=13, color=PALETTE["ink"])
bsp_fig_path = save_matplotlib(fig, TOPIC, "bsp-triangle-split-visibility-order.png")
close(fig)
display_artifact(bsp_fig_path, width=900)
display_artifact(bsp_check_path)



The split produces two children on one side and one child on the other. The area check is a simple but powerful guard: if the child areas do not add to the parent area, the split lost or duplicated surface. The "children span plane" check catches a different error: a child should not still need to be classified as crossing the same splitter.

The small visibility-order panel is the BSP traversal rule in data-structure form. The tree can be built once; only the sign of the eye point changes the back-to-front order.


## 5. Tiled and blocked arrays: locality is an indexing problem

A 2D array stored row-major makes horizontal neighbors adjacent in memory, but vertical neighbors are separated by the row width. Tiling makes locality more symmetric by storing small square blocks contiguously. The indexing arithmetic decomposes an index into two parts: which tile contains the element, and where the element lies inside that tile.

The chapter also describes a two-level 3D blocked layout for volume data. The important computational pattern is the same: the final address can be decomposed into per-axis lookup tables, `Fx(x) + Fy(y)` in 2D or `Fx(x) + Fy(y) + Fz(z)` in 3D. That table form can replace repeated integer divide and modulus operations in tight loops.



In [ ]:
def row_major_index(x: int, y: int, nx: int) -> int:
    return x + nx * y


def padded_size(n: int, tile: int) -> int:
    return int(math.ceil(n / tile) * tile)


def tiled_index_2d(x: int, y: int, nx: int, ny: int, tile: int) -> int:
    px = padded_size(nx, tile)
    bx, lx = divmod(x, tile)
    by, ly = divmod(y, tile)
    tiles_per_row = px // tile
    return tile * tile * (tiles_per_row * by + bx) + ly * tile + lx


def tiled_lookup_tables(nx: int, ny: int, tile: int) -> tuple[np.ndarray, np.ndarray]:
    px = padded_size(nx, tile)
    tiles_per_row = px // tile
    Fx = np.array([tile * tile * (x // tile) + (x % tile) for x in range(nx)], dtype=int)
    Fy = np.array([tile * tile * tiles_per_row * (y // tile) + (y % tile) * tile for y in range(ny)], dtype=int)
    return Fx, Fy


def vertical_scan_indices(nx: int, ny: int, indexer) -> np.ndarray:
    return np.array([indexer(x, y) for x in range(nx) for y in range(ny)], dtype=int)


def horizontal_scan_indices(nx: int, ny: int, indexer) -> np.ndarray:
    return np.array([indexer(x, y) for y in range(ny) for x in range(nx)], dtype=int)


def mean_abs_stride(indices: np.ndarray) -> float:
    return float(np.mean(np.abs(np.diff(indices))))

nx_size, ny_size, tile = 8, 6, 3
row_order = np.array([[row_major_index(x, y, nx_size) for x in range(nx_size)] for y in range(ny_size)])
tile_order = np.array([[tiled_index_2d(x, y, nx_size, ny_size, tile) for x in range(nx_size)] for y in range(ny_size)])
Fx, Fy = tiled_lookup_tables(nx_size, ny_size, tile)
lookup_equal = all(tiled_index_2d(x, y, nx_size, ny_size, tile) == int(Fx[x] + Fy[y]) for x in range(nx_size) for y in range(ny_size))

row_vertical = vertical_scan_indices(nx_size, ny_size, lambda x, y: row_major_index(x, y, nx_size))
tile_vertical = vertical_scan_indices(nx_size, ny_size, lambda x, y: tiled_index_2d(x, y, nx_size, ny_size, tile))
row_horizontal = horizontal_scan_indices(nx_size, ny_size, lambda x, y: row_major_index(x, y, nx_size))
tile_horizontal = horizontal_scan_indices(nx_size, ny_size, lambda x, y: tiled_index_2d(x, y, nx_size, ny_size, tile))

sweep_rows = []
for t in range(1, 7):
    sweep_rows.append(
        {
            "tile_size": t,
            "padded_x": padded_size(nx_size, t),
            "padded_y": padded_size(ny_size, t),
            "row_major_vertical_mean_stride": mean_abs_stride(row_vertical),
            "tiled_vertical_mean_stride": mean_abs_stride(vertical_scan_indices(nx_size, ny_size, lambda x, y, t=t: tiled_index_2d(x, y, nx_size, ny_size, t))),
            "tiled_horizontal_mean_stride": mean_abs_stride(horizontal_scan_indices(nx_size, ny_size, lambda x, y, t=t: tiled_index_2d(x, y, nx_size, ny_size, t))),
        }
    )
stride_sweep_path = save_table_csv(sweep_rows, TOPIC, "tiled-stride-sweep.csv")


def blocked_index_3d(x: int, y: int, z: int, nx: int, ny: int, nz: int, n: int, m: int) -> int:
    bx, lx = divmod(x, n)
    by, ly = divmod(y, n)
    bz, lz = divmod(z, n)
    mx, ibx = divmod(bx, m)
    my, iby = divmod(by, m)
    mz, ibz = divmod(bz, m)
    B_y = (ny // n) // m
    B_z = (nz // n) // m
    macro = (mx * B_y * B_z + my * B_z + mz) * (n ** 3) * (m ** 3)
    brick = (ibx * m * m + iby * m + ibz) * (n ** 3)
    local = lx * n * n + ly * n + lz
    return macro + brick + local


def blocked_lookup_3d(nx: int, ny: int, nz: int, n: int, m: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    B_y = (ny // n) // m
    B_z = (nz // n) // m
    Fx3 = []
    for x in range(nx):
        bx, lx = divmod(x, n)
        mx, ibx = divmod(bx, m)
        Fx3.append(mx * (n ** 3) * (m ** 3) * B_y * B_z + ibx * (n ** 3) * m * m + lx * n * n)
    Fy3 = []
    for y in range(ny):
        by, ly = divmod(y, n)
        my, iby = divmod(by, m)
        Fy3.append(my * (n ** 3) * (m ** 3) * B_z + iby * (n ** 3) * m + ly * n)
    Fz3 = []
    for z in range(nz):
        bz, lz = divmod(z, n)
        mz, ibz = divmod(bz, m)
        Fz3.append(mz * (n ** 3) * (m ** 3) + ibz * (n ** 3) + lz)
    return np.array(Fx3), np.array(Fy3), np.array(Fz3)

nx3, ny3, nz3, n3, m3 = 40, 20, 20, 5, 2
Fx3, Fy3, Fz3 = blocked_lookup_3d(nx3, ny3, nz3, n3, m3)
test_points = [(0, 0, 0), (4, 4, 4), (5, 0, 0), (13, 7, 8), (39, 19, 19)]
blocked_lookup_equal = all(blocked_index_3d(x, y, z, nx3, ny3, nz3, n3, m3) == int(Fx3[x] + Fy3[y] + Fz3[z]) for x, y, z in test_points)

locality_checks = {
    "nx": nx_size,
    "ny": ny_size,
    "tile": tile,
    "padded_x": padded_size(nx_size, tile),
    "padded_y": padded_size(ny_size, tile),
    "lookup_formula_matches_direct_2d": bool(lookup_equal),
    "row_major_vertical_mean_stride": mean_abs_stride(row_vertical),
    "tiled_vertical_mean_stride": mean_abs_stride(tile_vertical),
    "row_major_horizontal_mean_stride": mean_abs_stride(row_horizontal),
    "tiled_horizontal_mean_stride": mean_abs_stride(tile_horizontal),
    "vertical_stride_improved_by_tiling": bool(mean_abs_stride(tile_vertical) < mean_abs_stride(row_vertical)),
    "blocked_3d_lookup_matches_direct_formula": bool(blocked_lookup_equal),
    "blocked_3d_test_points": test_points,
}
locality_check_path = save_json(locality_checks, TOPIC, "tiled-array-locality-checks.json")
locality_checks



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.6), gridspec_kw={"width_ratios": [1.0, 1.0, 1.05]})
for ax, matrix, title in [
    (axes[0], row_order, "row-major address: index = x + Nx y"),
    (axes[1], tile_order, "3x3 tiled address with padding"),
]:
    im = ax.imshow(matrix, cmap="viridis")
    for y in range(matrix.shape[0]):
        for x in range(matrix.shape[1]):
            ax.text(x, y, str(matrix[y, x]), ha="center", va="center", color="white", fontsize=7)
    ax.set_xticks(range(nx_size))
    ax.set_yticks(range(ny_size))
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[2]
sweep_df = pd.DataFrame(sweep_rows)
ax.plot(sweep_df["tile_size"], sweep_df["tiled_vertical_mean_stride"], "o-", color=PALETTE["teal"], label="tiled vertical scan")
ax.axhline(mean_abs_stride(row_vertical), color=PALETTE["red"], linestyle="--", label="row-major vertical scan")
ax.plot(sweep_df["tile_size"], sweep_df["tiled_horizontal_mean_stride"], "s-", color=PALETTE["blue"], label="tiled horizontal scan")
style_axis(ax, "Applied lab: tile size changes locality", xlabel="tile size", ylabel="mean absolute memory stride")
ax.legend(fontsize=8)

fig.suptitle("Tiled arrays make locality more symmetric for multidimensional scans", fontsize=13, color=PALETTE["ink"])
tiling_fig_path = save_matplotlib(fig, TOPIC, "tiled-array-locality-scan.png")
close(fig)
display_artifact(tiling_fig_path, width=920)
display_artifact(locality_check_path)
display_artifact(stride_sweep_path)



The heatmaps label the actual one-dimensional memory addresses. In row-major order, moving vertically jumps by `Nx`. In a tiled layout, short vertical moves inside a tile are close in memory, and the larger jumps occur only at tile boundaries. The CSV sweep is the applied lab: change the tile size and decide whether the new padding and stride pattern are better for the scan you care about.

For the 3D blocked layout, the notebook checks the same decomposition as the chapter's two-level volume example: a direct address computation equals `Fx(x) + Fy(y) + Fz(z)` on representative points.


## Applied lab

Use one of these small changes and predict which check should move before re-running the cells:

1. **Mesh surgery:** flip one face orientation in `faces`. The visual still draws a pyramid, but the winding and opposite-directed-edge checks should fail.
2. **Scene graph edit:** move the car node relative to the ferry. Wheel centers should move with the car, while the ferry and mast remain controlled by their own parent path.
3. **Traversal design:** rotate the ray in the spatial section. The first hit may change, and the BVH/grid candidate counts should change with the path through boxes and cells.
4. **BSP tolerance:** lower the epsilon in `split_triangle_by_line` and use a near-plane vertex. Watch for a tiny sliver child and an area/check pattern that explains why robust classification matters.
5. **Array locality:** change `tile` or the scan order. A tile size is not universally good; it is good for a memory unit and a traversal pattern.

The lab's standard is not "the picture changed." A good answer names the data-structure query, the visual change, and the invariant or count that changed with it.

## Sanity checks

The final cell checks the core identities from the chapter-specific examples, verifies generated artifacts exist and are nonempty, and writes a final JSON report under `artifacts/chapter-12/checks/`.



In [ ]:
all_artifacts = [
    source_span_path,
    storyboard_path,
    mesh_fig_path,
    mesh_check_path,
    scene_fig_path,
    scene_check_path,
    spatial_fig_path,
    spatial_html_path,
    spatial_check_path,
    traversal_table_path,
    bsp_fig_path,
    bsp_check_path,
    tiling_fig_path,
    locality_check_path,
    stride_sweep_path,
]
artifact_records = assert_artifacts(all_artifacts)
image_records = [assert_nonblank_image(path) for path in [mesh_fig_path, scene_fig_path, spatial_fig_path, bsp_fig_path, tiling_fig_path]]

assert mesh_checks["euler_characteristic"] == 2
assert mesh_checks["boundary_edge_count"] == 0
assert mesh_checks["nonmanifold_edge_count"] == 0
assert mesh_checks["halfedge_pair_reciprocal"]
assert mesh_checks["halfedge_face_next_cycles"]
assert scene_checks["stack_restored_after_traversal"]
assert scene_checks["front_left_wheel_explicit_product_match"]
assert spatial_checks["first_hit_brute_force"][1] == spatial_checks["first_hit_bvh"][1] == spatial_checks["first_hit_grid"][1]
assert spatial_checks["bvh_primitive_tests"] < spatial_checks["brute_force_primitive_tests"]
assert spatial_checks["grid_candidate_tests"] <= spatial_checks["brute_force_primitive_tests"]
assert bsp_checks["area_error"] < 1e-10
assert not bsp_checks["children_span_plane"]
assert locality_checks["lookup_formula_matches_direct_2d"]
assert locality_checks["vertical_stride_improved_by_tiling"]
assert locality_checks["blocked_3d_lookup_matches_direct_formula"]

final_report = {
    "chapter": CHAPTER,
    "title": "Data Structures for Graphics",
    "source_span": {"printed_pages": "291-334", "pdf_pages": "308-351"},
    "artifact_count": len(all_artifacts),
    "nonblank_image_count": len(image_records),
    "checks": {
        "mesh": mesh_checks,
        "scene_graph": scene_checks,
        "spatial": spatial_checks,
        "bsp": bsp_checks,
        "tiled_arrays": locality_checks,
    },
    "libraries_used": ["numpy", "matplotlib", "networkx", "plotly", "trimesh", "pandas"],
}
final_path = save_json(final_report, TOPIC, "final-sanity.json")
display_artifact(final_path)
final_report



## Takeaways

- Meshes are not just bags of triangles. Shared vertices reduce storage, and connectivity structures make local topology queries cheap.
- Scene graphs store local transforms; traversal composes them into world transforms only when an object is visited.
- Bounding boxes, BVHs, grids, and BSP trees all reduce geometric search, but they partition different things: objects, space, or signed half-spaces.
- BSP visibility works because a signed plane function gives a viewpoint-dependent far-to-near traversal order after crossing primitives have been split.
- Tiled arrays turn locality into integer indexing. The right tile size depends on the memory hierarchy and the scan pattern.
- The reliable way to use any of these structures is to pair the visual intuition with invariants: edge counts, stack products, interval hits, area conservation, and address formulas.
